# SpaceX Falcon 9 — Data Collection with the SpaceX REST API

This notebook collects Falcon 9 launch records using the public SpaceX REST API
(`api.spacexdata.com`). For every past launch we pull the rocket, launch pad,
payload and core (booster) details and assemble them into a single flat table
that later notebooks use for wrangling, SQL/visual EDA and predictive modeling.

In [1]:
import requests
import pandas as pd
import numpy as np
import datetime

pd.set_option('display.max_columns', None)
API = "https://api.spacexdata.com/v4"

## 1.1 Helper functions

For every launch we look up its rocket, launchpad, payload(s) and core(s) by id.

In [2]:
BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Outcome, Flights, GridFins, Reused, Legs = [], [], [], [], []
LandingPad, Block, ReusedCount, Serial, Longitude, Latitude = [], [], [], [], [], []

def getBoosterVersion(data):
    for x in data['rocket']:
        response = requests.get(f"{API}/rockets/{x}", timeout=10).json()
        BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        response = requests.get(f"{API}/launchpads/{x}", timeout=10).json()
        Longitude.append(response['longitude'])
        Latitude.append(response['latitude'])
        LaunchSite.append(response['name'])

def getPayloadData(data):
    for load in data['payloads']:
        response = requests.get(f"{API}/payloads/{load}", timeout=10).json()
        PayloadMass.append(response.get('mass_kg'))
        Orbit.append(response.get('orbit'))

def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get(f"{API}/cores/{core['core']}", timeout=10).json()
            Block.append(response.get('block'))
            ReusedCount.append(response.get('reuse_count'))
            Serial.append(response.get('serial'))
        else:
            Block.append(None); ReusedCount.append(None); Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

## 1.2 Pull the launch list and enrich each record

We request the full list of past launches, then call the helper functions above for every record.

In [3]:
api_reachable = False
try:
    resp = requests.get(f"{API}/launches/past", timeout=10)
    print("SpaceX API status code:", resp.status_code)
    api_reachable = resp.status_code == 200
except Exception as e:
    print("SpaceX API request failed:", repr(e))

if api_reachable:
    data = pd.json_normalize(resp.json())
    data = data[['rocket', 'payloads', 'launchpad', 'cores', 'flight_number', 'date_utc']]
    data = data[data['cores'].map(len) == 1]
    data = data[data['payloads'].map(len) == 1]
    data['cores'] = data['cores'].map(lambda x: x[0])
    data['payloads'] = data['payloads'].map(lambda x: x[0])
    data['date'] = pd.to_datetime(data['date_utc']).dt.date
    data = data[data['date'] <= datetime.date(2020, 11, 13)]

    getBoosterVersion(data)
    getLaunchSite(data)
    getPayloadData(data)
    getCoreData(data)

    launch_dict = {
        'FlightNumber': list(data['flight_number']), 'Date': list(data['date']),
        'BoosterVersion': BoosterVersion, 'PayloadMass': PayloadMass, 'Orbit': Orbit,
        'LaunchSite': LaunchSite, 'Outcome': Outcome, 'Flights': Flights, 'GridFins': GridFins,
        'Reused': Reused, 'Legs': Legs, 'LandingPad': LandingPad, 'Block': Block,
        'ReusedCount': ReusedCount, 'Serial': Serial, 'Longitude': Longitude, 'Latitude': Latitude,
    }
    df = pd.DataFrame(launch_dict)
    df = df[df['BoosterVersion'] == 'Falcon 9'].reset_index(drop=True)
    df['FlightNumber'] = list(range(1, df.shape[0] + 1))
    print("Collected", df.shape[0], "Falcon 9 launch records directly from the live API.")
else:
    print("api.spacexdata.com did not return a usable response at execution time")
    print("(community-run API — intermittently returns a Cloudflare 525 handshake error).")
    print("Falling back to the archived extract of this exact same collection step,")
    print("hosted by IBM Skills Network for this course:")
    df = pd.read_csv(f"https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
    print("Loaded", df.shape[0], "records from dataset_part_1.csv")

df.head()

SpaceX API status code: 525
api.spacexdata.com did not return a usable response at execution time
(community-run API — intermittently returns a Cloudflare 525 handshake error).
Falling back to the archived extract of this exact same collection step,
hosted by IBM Skills Network for this course:


Loaded 90 records from dataset_part_1.csv


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


In [4]:
print(df.shape)
df.isnull().sum()

(90, 17)


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

In [5]:
df.to_csv("dataset_part_1.csv", index=False)
print("Saved dataset_part_1.csv")

Saved dataset_part_1.csv
